# EDA - Analyse Exploratoire des Données
## Projet : Système Intelligent de Rétention Client
**Tâche prédictive** : Prédiction du Churn (Classification Binaire)

### Contexte
Le Machine Learning supervisé permet d'exploiter les données comportementales des clients pour
*« prédire la probabilité qu'un client résilie son abonnement »* (classification binaire).
Contrairement à une simple corrélation statistique, le ML *« apprend automatiquement des relations
complexes entre comportements clients et churn »* et *« capture des interactions non linéaires
invisibles à l'œil humain »* (cf. Cadre Théorique du sujet).

### Objectif de l'EDA
Comme le recommande le sujet : *« Avant toute modélisation, prenez le temps de comprendre vos
données : distributions, valeurs extrêmes, cohérence des variables, présence de valeurs manquantes,
éventuelles anomalies et relations entre variables. »*

Cette EDA vise à :
1. Auditer le dataset (valeurs manquantes, outliers, incohérences)
2. Analyser le déséquilibre des classes et ses implications sur les métriques
3. Identifier les variables discriminantes et les redondances
4. Construire une intuition métier pour orienter le feature engineering et le choix des modèles

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

FIGURES_DIR = '../reports/figures/'

df = pd.read_csv('../customer_churn.csv')
print(f'Shape: {df.shape}')
df.head()

## 1. Inspection Initiale

In [ ]:
print('=== Types des colonnes ===')
print(df.dtypes.to_string())
print(f'\nNombre de colonnes numériques: {df.select_dtypes(include="number").shape[1]}')
print(f'Nombre de colonnes catégorielles: {df.select_dtypes(include="object").shape[1]}')

In [ ]:
df.describe().round(2)

## 2. Valeurs Manquantes

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
missing_df = pd.DataFrame({'count': missing, 'pct': missing_pct})
missing_df = missing_df[missing_df['count'] > 0]

if len(missing_df) > 0:
    print('Colonnes avec valeurs manquantes:')
    print(missing_df)
    print(f'\n=> complaint_type manquant pour {missing_df.loc["complaint_type", "pct"]}% des clients')
    print('   Hypothèse : ces clients n\'ont pas déposé de plainte -> on encodera comme "No_Complaint"')
else:
    print('Aucune valeur manquante')

print(f'\nDoublons: {df.duplicated().sum()}')

## 3. Analyse de la Variable Cible : Churn

### Pourquoi le déséquilibre des classes est critique
Le sujet nous avertit : *« Dans un problème de churn, il est fréquent que la classe churn=1 soit
minoritaire. Un modèle peut donc afficher une accuracy élevée tout en étant inefficace pour détecter
les clients réellement à risque. »*

**Exemple concret** : si notre dataset contient 90% de non-churners, un modèle qui prédit **toujours**
"non-churn" sans rien apprendre obtiendra 90% d'accuracy. Pourtant il est totalement inutile car il
rate 100% des churners. En contexte business, *« les faux négatifs (churners non détectés) peuvent
coûter très cher »* — rater un client à forte valeur qui s'apprête à partir est bien plus coûteux que
de contacter par erreur un client fidèle.

C'est pourquoi le sujet impose d'utiliser des **métriques adaptées** :
- **Recall** (Rappel) : parmi tous les vrais churners, combien le modèle en détecte-t-il ? C'est la métrique la plus critique dans notre contexte.
- **Precision** : parmi tous les clients prédits comme churners, combien le sont réellement ?
- **F1-score** : moyenne harmonique de Precision et Recall — le compromis entre les deux.
- **ROC-AUC** : capacité globale du modèle à discriminer les classes, indépendamment du seuil de décision.
- **PR-AUC** (Precision-Recall AUC) : *« fortement recommandé si déséquilibre important »* — plus informative que la ROC-AUC en cas de forte asymétrie.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Distribution
churn_counts = df['churn'].value_counts()
colors = ['#2ecc71', '#e74c3c']
axes[0].bar(['Non-Churn (0)', 'Churn (1)'], churn_counts.values, color=colors)
for i, v in enumerate(churn_counts.values):
    axes[0].text(i, v + 50, str(v), ha='center', fontweight='bold')
axes[0].set_title('Distribution du Churn')
axes[0].set_ylabel('Nombre de clients')

# Proportions
axes[1].pie(churn_counts.values, labels=['Non-Churn', 'Churn'],
            colors=colors, autopct='%1.1f%%', startangle=90,
            explode=(0, 0.1), textprops={'fontsize': 12})
axes[1].set_title('Proportion du Churn')

plt.suptitle(f'Déséquilibre des classes — Ratio churn: {df["churn"].mean():.1%}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}churn_distribution.png', bbox_inches='tight')
plt.show()

print(f'Ratio de déséquilibre: 1:{churn_counts[0]//churn_counts[1]} (majoritaire vs minoritaire)')
print('\n=> Un modèle naïf prédisant toujours "non-churn" obtiendrait {:.1%} d\'accuracy'.format(churn_counts[0]/len(df)))
print('   mais raterait 100% des churners. C\'est pourquoi l\'accuracy seule est trompeuse.')
print('\n=> Métriques adaptées : Recall (détection des churners), F1 (compromis),')  
print('   ROC-AUC et PR-AUC (performance globale indépendante du seuil).')
print('\n=> Stratégies de gestion du déséquilibre à tester :')
print('   - SMOTE (suréchantillonnage synthétique de la classe minoritaire)')
print('   - class_weight="balanced" (pondération des classes dans le modèle)')
print('   - Ajustement du seuil de décision (ne pas se limiter à 0.5)')

## 4. Distributions des Variables Numériques

L'objectif est d'identifier visuellement quelles variables ont des **distributions différentes** entre
churners et non-churners. Une variable dont les histogrammes se superposent parfaitement n'apporte
pas d'information discriminante. À l'inverse, si les courbes sont décalées, la variable est un signal
potentiel pour le modèle.

In [ ]:
num_cols = ['age', 'tenure_months', 'monthly_logins', 'weekly_active_days',
            'avg_session_time', 'features_used', 'usage_growth_rate',
            'last_login_days_ago', 'monthly_fee', 'total_revenue',
            'payment_failures', 'support_tickets', 'avg_resolution_time',
            'csat_score', 'escalations', 'email_open_rate',
            'marketing_click_rate', 'nps_score', 'referral_count']

fig, axes = plt.subplots(5, 4, figsize=(20, 22))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    ax = axes[i]
    df[df['churn']==0][col].hist(ax=ax, bins=30, alpha=0.6, color='#2ecc71', label='Non-Churn', density=True)
    df[df['churn']==1][col].hist(ax=ax, bins=30, alpha=0.6, color='#e74c3c', label='Churn', density=True)
    ax.set_title(col, fontsize=10)
    ax.legend(fontsize=7)

# Masquer le dernier subplot vide
axes[-1].set_visible(False)

plt.suptitle('Distributions des variables numériques par statut de Churn', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}distributions_numeriques.png', bbox_inches='tight')
plt.show()

## 5. Variables Catégorielles vs Churn

### Qu'est-ce que « discriminer » le churn ?
Une variable **discrimine** le churn si sa valeur permet de **séparer** les churners des non-churners.
Concrètement, si le taux de churn est quasiment identique quelle que soit la catégorie (ex: Homme ~10%,
Femme ~10%), alors la variable n'apporte pas d'information utile pour la prédiction — elle ne discrimine
pas. À l'inverse, si une catégorie avait un taux de churn de 25% et une autre de 5%, la variable serait
fortement discriminante.

In [ ]:
cat_cols = ['gender', 'customer_segment', 'signup_channel', 'contract_type',
            'payment_method', 'discount_applied', 'price_increase_last_3m',
            'complaint_type', 'survey_response']

fig, axes = plt.subplots(3, 3, figsize=(18, 14))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    ax = axes[i]
    churn_rate = df.groupby(col)['churn'].mean().sort_values(ascending=False)
    bars = ax.bar(range(len(churn_rate)), churn_rate.values, color='#3498db')
    ax.set_xticks(range(len(churn_rate)))
    ax.set_xticklabels(churn_rate.index, rotation=45, ha='right', fontsize=8)
    ax.set_title(f'Taux de churn par {col}', fontsize=10)
    ax.set_ylabel('Taux de churn')
    ax.axhline(y=df['churn'].mean(), color='red', linestyle='--', alpha=0.7, label=f'Moyenne ({df["churn"].mean():.1%})')
    ax.legend(fontsize=7)
    # Valeurs sur les barres
    for bar, val in zip(bars, churn_rate.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                f'{val:.1%}', ha='center', fontsize=7)

plt.suptitle('Taux de Churn par variable catégorielle', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}churn_rate_categorielles.png', bbox_inches='tight')
plt.show()

print('Observation : les variables catégorielles ne discriminent pas significativement le churn.')
print('Le taux de churn reste entre 9% et 11% quelle que soit la modalité.')
print('=> Cela signifie que connaître le genre, le type de contrat ou le pays d\'un client')
print('   ne change quasiment pas sa probabilité de churn.')
print('=> Les variables numériques comportementales (CSAT, logins, payment_failures)')
print('   seront bien plus discriminantes pour le modèle.')

## 6. Matrice de Corrélation

Le sujet recommande : *« L'étude des corrélations et des relations entre features est importante pour
comprendre la structure des données, identifier les variables redondantes et limiter certains risques
(multicolinéarité, surinterprétation de variables très liées). »*

La **multicolinéarité** est le fait que deux variables mesurent quasiment la même chose (ex: total_revenue
est le produit de monthly_fee × tenure_months). Cela peut :
- Rendre instables les coefficients d'une régression logistique
- Fausser l'interprétation de l'importance des variables

L'objectif ici est d'identifier ces redondances pour décider si on garde, supprime ou transforme certaines variables.

In [ ]:
corr_cols = num_cols + ['churn']
corr_matrix = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(16, 13))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=ax, annot_kws={'size': 7},
            linewidths=0.5)
ax.set_title('Matrice de Corrélation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}matrice_correlation.png', bbox_inches='tight')
plt.show()

print('Corrélations notables:')
print('  - monthly_fee <-> total_revenue: 0.71 (redondance potentielle)')
print('  - tenure_months <-> total_revenue: 0.59 (logique: plus longtemps = plus de revenus)')
print('  - csat_score <-> churn: -0.16 (plus le score est haut, moins de churn)')
print('  - tenure_months <-> churn: -0.12 (clients fidèles churnent moins)')
print('  - payment_failures <-> churn: +0.11 (échecs de paiement = signal de churn)')

## 7. Corrélation avec le Churn (classement)

### Pourquoi des corrélations faibles justifient des modèles non-linéaires
La corrélation de Pearson ne mesure que les **relations linéaires**. Une corrélation faible ne signifie
pas que la variable est inutile — elle peut avoir une relation **non-linéaire** ou **conditionnelle** avec
le churn (ex: un client avec peu de logins ET des échecs de paiement ET un faible CSAT a un risque
élevé, mais aucune de ces variables n'est fortement corrélée au churn individuellement).

C'est ce que le sujet décrit : *« Le Machine Learning capture des interactions non linéaires invisibles
à l'œil humain »* et *« s'adapte à des profils clients hétérogènes »*.

D'où la nécessité de modèles capables de capturer ces interactions :
- **Logistic Regression** (baseline) : modèle linéaire, simple et interprétable, mais limité aux
  relations linéaires. Servira de point de référence.
- **Random Forest** : ensemble de centaines d'arbres de décision indépendants qui « votent ».
  Comme le dit le sujet : *« Un Random Forest peut capturer des comportements non linéaires. »*
  Chaque arbre apprend des combinaisons de variables différentes.
- **XGBoost (Gradient Boosting)** : aussi basé sur des arbres, mais chaque arbre corrige
  séquentiellement les erreurs du précédent. Le sujet indique : *« Un Gradient Boosting peut
  offrir de meilleures performances prédictives. »* C'est souvent le plus performant sur données
  tabulaires.

Cette progression suit la recommandation du sujet : *« Commencez par un modèle baseline simple,
puis complexifiez progressivement. »*

In [ ]:
corr_churn = df[num_cols].corrwith(df['churn']).sort_values()

fig, ax = plt.subplots(figsize=(10, 8))
colors = ['#e74c3c' if v > 0 else '#2ecc71' for v in corr_churn.values]
ax.barh(corr_churn.index, corr_churn.values, color=colors)
ax.set_xlabel('Corrélation avec Churn')
ax.set_title('Corrélation linéaire des variables avec le Churn', fontsize=14, fontweight='bold')
ax.axvline(x=0, color='black', linestyle='-', linewidth=0.5)

for i, (val, name) in enumerate(zip(corr_churn.values, corr_churn.index)):
    ax.text(val + (0.002 if val > 0 else -0.002), i, f'{val:.3f}',
            va='center', ha='left' if val > 0 else 'right', fontsize=8)

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}correlation_churn.png', bbox_inches='tight')
plt.show()

print('Note : les corrélations linéaires (Pearson) avec le churn sont toutes faibles (< 0.2).')
print('Cela ne signifie PAS que ces variables sont inutiles.')
print('Cela signifie que la relation entre chaque variable et le churn n\'est pas linéaire,')
print('ou qu\'elle dépend de combinaisons de plusieurs variables (interactions).')
print('\nExemple : un client avec peu de logins ET des échecs de paiement ET un faible CSAT')
print('a un risque élevé, même si aucune variable seule n\'est fortement corrélée au churn.')
print('\n=> C\'est précisément pourquoi nous utiliserons des modèles non-linéaires')
print('   (Random Forest, XGBoost) capables de capturer ces interactions.')

## 8. Analyse des Outliers (Boxplots)

Les outliers (valeurs extrêmes) peuvent être de deux natures :
- Des **erreurs de données** (saisie, capteur défaillant) → à corriger ou supprimer
- Des **valeurs métier réalistes** mais rares → à conserver car elles portent de l'information

Le sujet recommande de repérer *« des problèmes invisibles au premier regard (variables constantes,
unités incohérentes, valeurs aberrantes, etc.) qui peuvent dégrader fortement les performances. »*

Notre analyse doit déterminer si les outliers détectés sont des signaux métier ou du bruit.

In [ ]:
outlier_cols = ['last_login_days_ago', 'monthly_fee', 'total_revenue',
                'payment_failures', 'csat_score', 'features_used']

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(outlier_cols):
    ax = axes[i]
    data = [df[df['churn']==0][col].dropna(), df[df['churn']==1][col].dropna()]
    bp = ax.boxplot(data, labels=['Non-Churn', 'Churn'], patch_artist=True)
    bp['boxes'][0].set_facecolor('#2ecc71')
    bp['boxes'][1].set_facecolor('#e74c3c')
    ax.set_title(col, fontsize=11)

plt.suptitle('Boxplots des variables avec outliers — par Churn', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}boxplots_outliers.png', bbox_inches='tight')
plt.show()

## 9. Analyse de Redondance : total_revenue vs monthly_fee / tenure

Le sujet souligne : *« Parfois, une variable dérivée est plus pertinente qu'une variable brute
(ex. ratio support_tickets / tenure_months, évolution d'usage via usage_growth_rate). L'objectif
n'est pas de supprimer mécaniquement des variables, mais de comprendre leur rôle dans le
comportement client. »*

Ici, `total_revenue` est quasi-déterminé par `monthly_fee × tenure_months`. Garder les trois
crée de la **multicolinéarité**, ce qui peut fausser l'interprétabilité des modèles linéaires.
On explorera la création de features dérivées (revenue_per_month, support_tickets_per_tenure, etc.).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

scatter_kw = dict(alpha=0.3, s=10)
for label, color in [(0, '#2ecc71'), (1, '#e74c3c')]:
    subset = df[df['churn'] == label]
    axes[0].scatter(subset['monthly_fee'], subset['total_revenue'], c=color, label=f'Churn={label}', **scatter_kw)
    axes[1].scatter(subset['tenure_months'], subset['total_revenue'], c=color, label=f'Churn={label}', **scatter_kw)

axes[0].set_xlabel('Monthly Fee')
axes[0].set_ylabel('Total Revenue')
axes[0].set_title('Total Revenue vs Monthly Fee (r=0.71)')
axes[0].legend()

axes[1].set_xlabel('Tenure (mois)')
axes[1].set_ylabel('Total Revenue')
axes[1].set_title('Total Revenue vs Tenure (r=0.59)')
axes[1].legend()

plt.suptitle('Analyse de redondance entre variables', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}redondance_revenue.png', bbox_inches='tight')
plt.show()

print('total_revenue ≈ monthly_fee × tenure_months')
print('=> On pourra créer un feature "revenue_per_month" = total_revenue / tenure_months')
print('   et potentiellement dropper total_revenue pour limiter la multicolinéarité')

## 10. Top 5 variables discriminantes (t-test)

In [ ]:
from scipy import stats

churn_0 = df[df['churn'] == 0]
churn_1 = df[df['churn'] == 1]

results = []
for col in num_cols:
    t_stat, p_val = stats.ttest_ind(churn_0[col].dropna(), churn_1[col].dropna())
    results.append({'variable': col, 't_stat': abs(t_stat), 'p_value': p_val})

results_df = pd.DataFrame(results).sort_values('t_stat', ascending=False)
print('Variables les plus discriminantes (t-test) :')
print(results_df.head(10).to_string(index=False))

print('\n=> Les variables avec le t-stat le plus élevé (et p-value < 0.05)')
print('   sont les meilleures candidates pour prédire le churn.')

## 11. Synthèse EDA

### Constats principaux

| Aspect | Constat | Implication | Action |
|--------|---------|-------------|--------|
| **Déséquilibre** | 89.8% vs 10.2% (ratio ~9:1) | L'accuracy est trompeuse : un modèle naïf obtient 90% en prédisant toujours 0. *« Les faux négatifs (churners non détectés) peuvent coûter très cher »* | SMOTE, class_weight, ajustement du seuil. Métriques : F1, Recall, ROC-AUC, PR-AUC |
| **Valeurs manquantes** | complaint_type: 20.5% | Probablement des clients sans plainte — c'est une info métier, pas un défaut | Encoder comme catégorie "No_Complaint" |
| **Corrélations linéaires faibles** | Max |r| = 0.16 avec churn | Les relations churn ↔ variables sont **non-linéaires** ou **conditionnelles** (combinaisons de facteurs). Un modèle linéaire seul ne suffira pas. | Justifie le recours à RF et XGBoost qui *« capturent des interactions non linéaires invisibles à l'œil humain »* |
| **Redondance** | total_revenue corrélé à monthly_fee (0.71) et tenure (0.59) | Multicolinéarité → instabilité des coefficients en régression logistique, surinterprétation | Feature engineering : revenue_per_month = total_revenue / tenure_months |
| **Outliers** | Modérés : last_login_days_ago (4.7%), monthly_fee/total_revenue (5.1%) | Ce sont des **signaux métier réalistes** (clients inactifs, forfaits premium) et non des erreurs de saisie | Conserver : les modèles à base d'arbres (RF, XGBoost) ne sont pas sensibles aux outliers. Normalisation pour la régression logistique. |
| **Catégorielles** | Taux de churn entre 9% et 11% pour toutes les modalités | Aucune variable catégorielle ne **discrimine** fortement le churn : la valeur de la catégorie ne change quasiment pas la probabilité de churn | Encoder (OneHot/Ordinal) mais impact prédictif limité attendu |
| **customer_id** | Identifiant unique, 10 000 valeurs distinctes | Aucune valeur prédictive, risque d'overfitting si conservé | Supprimer des features |

### Variables les plus prometteuses pour prédire le churn
Comme le sujet l'indique : *« Une forte baisse de l'activité récente augmente le risque de churn,
des échecs de paiement répétés sont un signal critique, un faible NPS est un indicateur précurseur
de départ. »*

Nos données confirment ces intuitions métier :
1. **`csat_score`** (r=-0.16) — satisfaction client : les clients insatisfaits churnent davantage
2. **`tenure_months`** (r=-0.12) — ancienneté : les clients de longue date sont plus fidèles
3. **`payment_failures`** (r=+0.11) — échecs de paiement : signal critique de désengagement
4. **`monthly_logins`** (r=-0.10) — fréquence d'usage : moins d'activité = plus de risque
5. **`total_revenue`** (r=-0.07) — valeur client : les clients à faible revenu churnent plus

### Stratégie de modélisation
En suivant la recommandation du sujet (*« Commencez par un modèle baseline simple, puis
complexifiez progressivement »*) :

1. **Logistic Regression** (baseline) — *« robuste et interprétable »* selon le sujet. Point de référence.
2. **Random Forest** — *« peut capturer des comportements non linéaires »*. Ensemble d'arbres indépendants.
3. **XGBoost (Gradient Boosting)** — *« peut offrir de meilleures performances prédictives »*. Arbres séquentiels corrigeant les erreurs.

### Prochaine étape
Preprocessing pipeline (sklearn.Pipeline pour éviter le data leakage) + modélisation multi-algorithmes.